# BERT NER FINE-TUNING FOR DLP BENCHMARK

In [ ]:
import os
import json
import random
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
SEED = 42
BASE_MODEL = "indobenchmark/indobert-base-p1"
OUTPUT_DIR = "./bert_dlp_finetuned"
MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 500

In [ ]:
# label schema - explicit, complete label space
LABEL_SCHEMA = {
    "O": 0,           # Outside any entity
    "B-NIK": 1,       # Begin NIK (16-digit national ID)
    "I-NIK": 2,       # Inside NIK
    "B-PHONE": 3,     # Begin phone number
    "I-PHONE": 4,     # Inside phone number
    "B-CREDIT_CARD": 5,  # Begin credit card
    "I-CREDIT_CARD": 6,  # Inside credit card
    "B-EMAIL": 7,     # Begin email
    "I-EMAIL": 8,     # Inside email
    "B-NAME": 9,      # Begin person name
    "I-NAME": 10,     # Inside person name
    "B-BANK": 11,     # Begin bank account
    "I-BANK": 12,     # Inside bank account
}

ID2LABEL = {v: k for k, v in LABEL_SCHEMA.items()}
NUM_LABELS = len(LABEL_SCHEMA)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# DATA PREPARATION
def tokenize_and_align_labels(
    text: str,
    entities: List[Dict],
    tokenizer,
    max_length: int = MAX_LENGTH
) -> Dict:
    """
    Tokenize text and align entity labels with subword tokens.

    Args:
        text: Input text
        entities: List of {start, end, label, value} dicts
        tokenizer: HuggingFace tokenizer
        max_length: Maximum sequence length

    Returns:
        Dictionary with input_ids, attention_mask, labels
    """
    # Tokenize
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    # Initialize labels as O (outside)
    labels = [LABEL_SCHEMA["O"]] * max_length

    # Get character-to-token mapping
    offset_mapping = encoding["offset_mapping"][0].tolist()

    # Align entity labels with tokens
    for entity in entities:
        start_char = entity["start"]
        end_char = entity["end"]
        label = entity["label"]

        # Find tokens that overlap with entity span
        token_start = None
        token_end = None

        for idx, (token_start_char, token_end_char) in enumerate(offset_mapping):
            # Skip special tokens (offset = (0, 0))
            if token_start_char == 0 and token_end_char == 0:
                continue

            # Find first token that overlaps with entity start
            if token_start is None and token_start_char <= start_char < token_end_char:
                token_start = idx

            # Find last token that overlaps with entity end
            if token_start_char < end_char <= token_end_char:
                token_end = idx
                break

        # Assign B- and I- labels
        if token_start is not None:
            labels[token_start] = LABEL_SCHEMA[f"B-{label}"]

            if token_end is not None and token_end > token_start:
                for idx in range(token_start + 1, token_end + 1):
                    labels[idx] = LABEL_SCHEMA[f"I-{label}"]

    return {
        "input_ids": encoding["input_ids"][0],
        "attention_mask": encoding["attention_mask"][0],
        "labels": torch.tensor(labels, dtype=torch.long)
    }


def extract_entities_from_prompt(row: pd.Series) -> Tuple[str, List[Dict]]:
    """
    Extract entity annotations from dataset row.

    Args:
        row: DataFrame row with prompt and ground truth fields

    Returns:
        (text, entities) where entities is list of {start, end, label, value}
    """
    text = row["prompt"]
    entities = []

    # Helper to find entity position in text
    def find_entity_span(text: str, value: str) -> Optional[Tuple[int, int]]:
        """Find start and end character positions of value in text"""
        if pd.isna(value) or str(value) == "None":
            return None

        value_str = str(value).strip()
        # Try exact match first
        start = text.find(value_str)
        if start != -1:
            return (start, start + len(value_str))

        # Try normalized match (remove spaces/dashes)
        import re
        normalized_value = re.sub(r'[\s\-]', '', value_str)
        normalized_text = re.sub(r'[\s\-]', '', text)
        start_norm = normalized_text.find(normalized_value)

        if start_norm != -1:
            # Map back to original text position (approximate)
            # This is a simplification; production code needs better alignment
            return (start_norm, start_norm + len(normalized_value))

        return None

    # Extract each PII type
    pii_fields = [
        ("ground_truth_nik", "NIK", "has_nik"),
        ("ground_truth_phone", "PHONE", "has_phone"),
        ("ground_truth_cc", "CREDIT_CARD", "has_cc"),
        ("ground_truth_bank", "BANK", "has_bank"),
        ("ground_truth_email", "EMAIL", "has_email"),
    ]

    for gt_col, label, has_col in pii_fields:
        if row.get(has_col, False) and gt_col in row:
            value = row[gt_col]
            span = find_entity_span(text, value)

            if span:
                entities.append({
                    "start": span[0],
                    "end": span[1],
                    "label": label,
                    "value": str(value)
                })

    # Extract names (always present)
    # Note: This requires name to be in the prompt, which may not always be true
    # For production, you'd need better name extraction logic

    return text, entities


class NERDataset(Dataset):
    """PyTorch Dataset for NER training"""

    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.encodings["labels"][idx]
        }


def prepare_training_data(
    df_prompts: pd.DataFrame,
    tokenizer,
    test_size: float = 0.2
) -> Tuple[NERDataset, NERDataset]:
    """
    Prepare training and validation datasets from prompt dataset.

    Args:
        df_prompts: DataFrame with prompts and ground truth
        tokenizer: HuggingFace tokenizer
        test_size: Fraction for validation set

    Returns:
        (train_dataset, val_dataset)
    """
    logger.info("Preparing training data...")

    # Extract entities from all prompts
    all_encodings = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    skipped = 0
    for idx, row in tqdm(df_prompts.iterrows(), total=len(df_prompts), desc="Processing prompts"):
        text, entities = extract_entities_from_prompt(row)

        # Skip if no entities found
        if not entities:
            skipped += 1
            continue

        encoding = tokenize_and_align_labels(text, entities, tokenizer)

        all_encodings["input_ids"].append(encoding["input_ids"])
        all_encodings["attention_mask"].append(encoding["attention_mask"])
        all_encodings["labels"].append(encoding["labels"])

    logger.info(f"Processed {len(all_encodings['input_ids'])} samples (skipped {skipped} without entities)")

    # Stack tensors
    all_encodings["input_ids"] = torch.stack(all_encodings["input_ids"])
    all_encodings["attention_mask"] = torch.stack(all_encodings["attention_mask"])
    all_encodings["labels"] = torch.stack(all_encodings["labels"])

    # Split train/val
    indices = list(range(len(all_encodings["input_ids"])))
    train_idx, val_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=SEED
    )

    train_encodings = {
        "input_ids": all_encodings["input_ids"][train_idx],
        "attention_mask": all_encodings["attention_mask"][train_idx],
        "labels": all_encodings["labels"][train_idx]
    }

    val_encodings = {
        "input_ids": all_encodings["input_ids"][val_idx],
        "attention_mask": all_encodings["attention_mask"][val_idx],
        "labels": all_encodings["labels"][val_idx]
    }

    train_dataset = NERDataset(train_encodings)
    val_dataset = NERDataset(val_encodings)

    logger.info(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

    return train_dataset, val_dataset

In [ ]:
# TRAINING

def compute_metrics(pred):
    """Compute precision, recall, F1 for NER evaluation"""
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # Remove padding (-100 labels)
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute metrics (simplified - production would use seqeval)
    from sklearn.metrics import precision_recall_fscore_support

    # Flatten
    flat_preds = [p for sublist in true_predictions for p in sublist]
    flat_labels = [l for sublist in true_labels for l in sublist]

    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, flat_preds, average="weighted", zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


def train_ner_model(
    train_dataset: NERDataset,
    val_dataset: NERDataset,
    output_dir: str = OUTPUT_DIR
):
    """
    Fine-tune BERT for NER on DLP dataset.

    Args:
        train_dataset: Training dataset
        val_dataset: Validation dataset
        output_dir: Directory to save model
    """
    logger.info(f"Loading base model: {BASE_MODEL}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL_SCHEMA
    )

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        warmup_steps=WARMUP_STEPS,
        logging_dir=f"{output_dir}/logs",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        seed=SEED,
        fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    )

    # Data collator
    data_collator = DataCollatorForTokenClassification(tokenizer)

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Train
    logger.info("Starting training...")
    trainer.train()

    # Save final model
    logger.info(f"Saving model to {output_dir}")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Save label schema
    with open(f"{output_dir}/label_schema.json", "w") as f:
        json.dump({
            "label2id": LABEL_SCHEMA,
            "id2label": ID2LABEL
        }, f, indent=2)

    logger.info("Training complete!")

    return trainer

In [ ]:
if __name__ == "__main__":
    logger.info("BERT NER FINE-TUNING FOR DLP BENCHMARK")

    # Load prompt dataset
    logger.info("Loading prompt_dataset.csv...")
    df_prompts = pd.read_csv("prompt_dataset.csv")

    # Ensure proper data types
    for col in ['ground_truth_nik', 'ground_truth_phone', 'ground_truth_cc', 'ground_truth_bank']:
        if col in df_prompts.columns:
            df_prompts[col] = df_prompts[col].astype(str).str.replace(r'\.0$', '', regex=True)
            df_prompts[col] = df_prompts[col].replace(['nan', 'None', 'NaN'], None)

    logger.info(f"Loaded {len(df_prompts)} prompts")

    # Initialize tokenizer
    logger.info(f"Loading tokenizer: {BASE_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    # Prepare datasets
    train_dataset, val_dataset = prepare_training_data(df_prompts, tokenizer)

    # Train model
    trainer = train_ner_model(train_dataset, val_dataset)

    # Evaluate on validation set
    logger.info("Final evaluation on validation set:")
    metrics = trainer.evaluate()
    for key, value in metrics.items():
        logger.info(f"  {key}: {value:.4f}")

    logger.info("TRAINING COMPLETE")
    logger.info(f"Model saved to: {OUTPUT_DIR}")
    logger.info(f"Label schema: {OUTPUT_DIR}/label_schema.json")
    logger.info("\nTo use in benchmark:")
    logger.info(f"  INDOBERT_NER_MODEL = '{OUTPUT_DIR}'")